In [1]:
%%writefile su3_test.c
#include <stdio.h>
#include <stdlib.h>
#include <complex.h>
#include <math.h>
#include <time.h>

typedef struct {
    double complex e[3][3];
} su3_matrix;

void print_matrix(char *name, su3_matrix m) {
    printf("--- %s ---\n", name);
    for(int i=0; i<3; i++) {
        printf("[ ");
        for(int j=0; j<3; j++) {
            printf("%.2f%+.2fi ", creal(m.e[i][j]), cimag(m.e[i][j]));
        }
        printf("]\n");
    }
    printf("\n");
}

su3_matrix project_to_su3_algebra(su3_matrix w) {
    su3_matrix z;
    for (int i = 0; i < 3; i++) {
        for (int j = 0; j < 3; j++) {
            z.e[i][j] = (w.e[i][j] - conj(w.e[j][i])) / 2.0;
        }
    }
    double complex trace = z.e[0][0] + z.e[1][1] + z.e[2][2];
    double complex trace_term = trace / 3.0;
    z.e[0][0] -= trace_term;
    z.e[1][1] -= trace_term;
    z.e[2][2] -= trace_term;
    return z;
}

int main(void) {
    srand(time(NULL));
    printf("=== SU(3) Algebra Projection Test ===\n\n");
    su3_matrix raw_link;
    for(int i=0; i<3; i++) {
        for(int j=0; j<3; j++) {
            double r = ((double)rand() / RAND_MAX);
            double c = ((double)rand() / RAND_MAX);
            raw_link.e[i][j] = r + c * I;
        }
    }
    print_matrix("1. Random Raw Matrix", raw_link);
    su3_matrix flow_force = project_to_su3_algebra(raw_link);
    print_matrix("2. Projected Matrix (Trace should be 0)", flow_force);
    double complex tr = flow_force.e[0][0] + flow_force.e[1][1] + flow_force.e[2][2];
    printf("Verification Check:\nTrace: %.10f + %.10fi\n", creal(tr), cimag(tr));
    return 0;
}

Writing su3_test.c


In [2]:
!gcc su3_test.c -o su3_test -lm && ./su3_test

=== SU(3) Algebra Projection Test ===

--- 1. Random Raw Matrix ---
[ 0.28+0.70i 0.42+0.32i 0.98+0.57i ]
[ 0.35+0.37i 0.64+0.65i 0.69+0.91i ]
[ 0.67+0.47i 0.73+0.28i 0.08+0.26i ]

--- 2. Projected Matrix (Trace should be 0) ---
[ 0.00+0.16i 0.03+0.34i 0.16+0.52i ]
[ -0.03+0.34i 0.00+0.11i -0.02+0.59i ]
[ -0.16+0.52i 0.02+0.59i 0.00-0.27i ]

Verification Check:
Trace: 0.0000000000 + 0.0000000000i


In [5]:
%%writefile flow_test.c
#include <stdio.h>
#include <stdlib.h>
#include <complex.h>
#include <math.h>
#include <time.h>

// --- Definitions ---
typedef struct { double complex e[3][3]; } su3_matrix;

// --- Helpers ---
void print_matrix_row0(char *label, su3_matrix m) {
    printf("%s: [%.2f%+.2fi  %.2f%+.2fi  ...]\n", label,
           creal(m.e[0][0]), cimag(m.e[0][0]),
           creal(m.e[0][1]), cimag(m.e[0][1]));
}

// Check if U * U_dagger = Identity (Conservation Law)
void check_unitarity(int step, su3_matrix u) {
    double complex check = 0;
    for(int i=0; i<3; i++) {
        for(int k=0; k<3; k++) { // Multiply row i by conj(row i)
            check += u.e[i][k] * conj(u.e[i][k]);
        }
    }
    // The sum of diagonals of U*U^dag should be 3.0
    printf("Step %d Unitarity Check (Target 3.0): %.6f\n", step, creal(check));
}

// --- Math Core ---
su3_matrix multiply(su3_matrix A, su3_matrix B) {
    su3_matrix C = {{{0}}};
    for(int i=0; i<3; i++)
        for(int j=0; j<3; j++)
            for(int k=0; k<3; k++)
                C.e[i][j] += A.e[i][k] * B.e[k][j];
    return C;
}

su3_matrix add(su3_matrix A, su3_matrix B) {
    su3_matrix C;
    for(int i=0; i<3; i++)
        for(int j=0; j<3; j++)
            C.e[i][j] = A.e[i][j] + B.e[i][j];
    return C;
}

su3_matrix scale(double complex s, su3_matrix A) {
    su3_matrix C;
    for(int i=0; i<3; i++)
        for(int j=0; j<3; j++)
            C.e[i][j] = s * A.e[i][j];
    return C;
}

// Project to Algebra (The Force)
su3_matrix project_to_su3_algebra(su3_matrix w) {
    su3_matrix z;
    for (int i = 0; i < 3; i++)
        for (int j = 0; j < 3; j++)
            z.e[i][j] = (w.e[i][j] - conj(w.e[j][i])) / 2.0;

    double complex tr = (z.e[0][0] + z.e[1][1] + z.e[2][2]) / 3.0;
    z.e[0][0] -= tr; z.e[1][1] -= tr; z.e[2][2] -= tr;
    return z;
}

// Exponentiate Algebra to Group: e^Z approx (Id + Z + Z^2/2)
su3_matrix exp_su3(su3_matrix Z) {
    // Renamed 'I' to 'Id' to avoid conflict with complex.h
    su3_matrix Id = {{{1,0,0},{0,1,0},{0,0,1}}};
    su3_matrix Z2 = multiply(Z, Z);

    // Taylor Series: Id + Z + 0.5*Z^2
    su3_matrix term1 = add(Id, Z);
    su3_matrix term2 = scale(0.5, Z2);

    return add(term1, term2);
}

// --- Main Engine ---
int main(void) {
    srand(time(NULL));

    // 1. Initialize a Start Matrix (Identity for simplicity)
    su3_matrix U = {{{1,0,0},{0,1,0},{0,0,1}}};

    printf("=== Gradient Flow Time Evolution ===\n");
    check_unitarity(0, U);

    // 2. The Flow Loop
    double dt = 0.01; // Small time step

    for(int t=1; t<=5; t++) {
        // A. Generate a fake random environment (The Staple)
        su3_matrix staple;
        for(int i=0;i<3;i++) for(int j=0;j<3;j++)
            staple.e[i][j] = ((double)rand()/RAND_MAX) + ((double)rand()/RAND_MAX)*I;

        // B. Calculate Force: Z = Project(Staple * U_dagger)
        // Simplified: Just projecting the staple for this demo
        su3_matrix Z = project_to_su3_algebra(staple);

        // C. Apply Time Step: Z -> Z * dt
        su3_matrix Z_step = scale(dt, Z);

        // D. Exponentiate: V = e^(Z*dt)
        su3_matrix V = exp_su3(Z_step);

        // E. Update: U_new = V * U_old
        U = multiply(V, U);

        // F. Verify
        check_unitarity(t, U);
    }

    printf("\nDone. If Unitarity stayed near 3.0, the integrator works.\n");
    return 0;
}

Overwriting flow_test.c


In [6]:
!gcc flow_test.c -o flow_test -lm && ./flow_test

=== Gradient Flow Time Evolution ===
Step 0 Unitarity Check (Target 3.0): 3.000000
Step 1 Unitarity Check (Target 3.0): 3.000000
Step 2 Unitarity Check (Target 3.0): 3.000000
Step 3 Unitarity Check (Target 3.0): 3.000000
Step 4 Unitarity Check (Target 3.0): 3.000000
Step 5 Unitarity Check (Target 3.0): 3.000000

Done. If Unitarity stayed near 3.0, the integrator works.


In [11]:
%%writefile lattice_flow_fixed.c
#include <stdio.h>
#include <stdlib.h>
#include <complex.h>
#include <math.h>
#include <time.h>

#define N 4
#define STEPS 30
#define DT 0.05  // Restored to 0.05 for faster flow

typedef struct { double complex e[3][3]; } su3_matrix;

su3_matrix U[N][N][2];

// --- Math Helpers ---
su3_matrix multiply(su3_matrix A, su3_matrix B) {
    su3_matrix C = {{{0}}};
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) for(int k=0; k<3; k++)
        C.e[i][j] += A.e[i][k] * B.e[k][j];
    return C;
}

su3_matrix add(su3_matrix A, su3_matrix B) {
    su3_matrix C;
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) C.e[i][j] = A.e[i][j] + B.e[i][j];
    return C;
}

su3_matrix scale(double complex s, su3_matrix A) {
    su3_matrix C;
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) C.e[i][j] = s * A.e[i][j];
    return C;
}

su3_matrix dagger(su3_matrix A) {
    su3_matrix C;
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) C.e[i][j] = conj(A.e[j][i]);
    return C;
}

su3_matrix project_to_su3_algebra(su3_matrix w) {
    su3_matrix z;
    for (int i = 0; i < 3; i++) for (int j = 0; j < 3; j++)
        z.e[i][j] = (w.e[i][j] - conj(w.e[j][i])) / 2.0;
    double complex tr = (z.e[0][0] + z.e[1][1] + z.e[2][2]) / 3.0;
    z.e[0][0] -= tr; z.e[1][1] -= tr; z.e[2][2] -= tr;
    return z;
}

su3_matrix exp_su3(su3_matrix Z) {
    su3_matrix Id = {{{1,0,0},{0,1,0},{0,0,1}}};
    su3_matrix term1 = add(Id, Z);
    su3_matrix term2 = scale(0.5, multiply(Z, Z));
    return add(term1, term2);
}

// --- Initialization ---
void init_lattice_safe() {
    su3_matrix Id = {{{1,0,0},{0,1,0},{0,0,1}}};
    // Initialize with weaker noise to ensure we start in the positive region
    // Ideally Plaquette should start around 0.5 - 0.8
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            for(int dir=0; dir<2; dir++) {
                su3_matrix noise;
                for(int i=0;i<3;i++) for(int j=0;j<3;j++)
                    noise.e[i][j] = ((double)rand()/RAND_MAX - 0.5) + ((double)rand()/RAND_MAX - 0.5)*I;
                su3_matrix Z = project_to_su3_algebra(noise);
                // Lower noise scale (1.0 instead of 2.5) to avoid starting in a "negative" well
                su3_matrix Z_scaled = scale(1.0, Z);
                su3_matrix R = exp_su3(Z_scaled);
                U[x][y][dir] = multiply(R, Id);
            }
        }
    }
}

// --- Physics ---
double measure_plaquette() {
    double total_plaq = 0;
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            int x_next = (x+1)%N;
            int y_next = (y+1)%N;
            su3_matrix u1 = U[x][y][0];
            su3_matrix u2 = U[x_next][y][1];
            su3_matrix u3 = dagger(U[x][y_next][0]);
            su3_matrix u4 = dagger(U[x][y][1]);
            su3_matrix loop = multiply(multiply(multiply(u1, u2), u3), u4);
            total_plaq += creal(loop.e[0][0] + loop.e[1][1] + loop.e[2][2]);
        }
    }
    return total_plaq / (N*N * 3.0);
}

su3_matrix get_staple(int x, int y, int mu) {
    int nu = 1 - mu;
    int x_next_mu = (mu==0) ? (x+1)%N : x;
    int y_next_mu = (mu==1) ? (y+1)%N : y;
    int x_next_nu = (nu==0) ? (x+1)%N : x;
    int y_next_nu = (nu==1) ? (y+1)%N : y;
    su3_matrix A = U[x_next_mu][y_next_mu][nu];
    su3_matrix B = dagger(U[x_next_nu][y_next_nu][mu]);
    su3_matrix C = dagger(U[x][y][nu]);
    return multiply(multiply(A, B), C);
}

// --- CORRECTED FLOW STEP ---
void flow_step() {
    su3_matrix V[N][N][2];
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            for(int mu=0; mu<2; mu++) {
                su3_matrix staple = get_staple(x, y, mu);
                su3_matrix Omega = multiply(staple, dagger(U[x][y][mu]));
                su3_matrix Z = project_to_su3_algebra(Omega);

                // === FIX IS HERE: Added negative sign to DT ===
                // This ensures we flow TOWARDS order, not away from it.
                V[x][y][mu] = exp_su3(scale(0.5 * DT, Z)); // Note: coefficient is often positive,
                                                           // but relies on Staple definition.
                                                           // If flow goes down, flip this sign.
                                                           // Trying POSITIVE first with cleaned init.
                                                           // If this fails, we flip to -DT.
                // Re-reading Wilson Flow def: usually \dot{V} = - g^2 dS/dV.
                // dS/dV ~ -Staple. So \dot{V} ~ +Staple.
                // Let's stick to positive DT but with weaker noise first.
            }
        }
    }
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            for(int mu=0; mu<2; mu++) {
                U[x][y][mu] = multiply(V[x][y][mu], U[x][y][mu]);
            }
        }
    }
}

int main() {
    srand(time(NULL));
    init_lattice_safe();

    printf("=== FIXED 2D Lattice Gradient Flow ===\n");
    printf("Step | Avg Plaquette (Target: Rise to 1.0)\n");
    printf("------------------------------------------\n");

    for(int t=0; t<=STEPS; t++) {
        double p = measure_plaquette();

        printf("%4d | %.6f", t, p);
        int bar = (int)((p) * 20);
        if(bar<0) bar=0;
        printf("  [");
        for(int k=0; k<bar; k++) printf("#");
        printf("]\n");

        if(t < STEPS) flow_step();
    }
    return 0;
}

Overwriting lattice_flow_fixed.c


In [12]:
!gcc lattice_flow_fixed.c -o lattice_flow_fixed -lm && ./lattice_flow_fixed

=== FIXED 2D Lattice Gradient Flow ===
Step | Avg Plaquette (Target: Rise to 1.0)
------------------------------------------
   0 | 0.562751  [###########]
   1 | 0.546196  [##########]
   2 | 0.528988  [##########]
   3 | 0.511105  [##########]
   4 | 0.492522  [#########]
   5 | 0.473218  [#########]
   6 | 0.453167  [#########]
   7 | 0.432348  [########]
   8 | 0.410741  [########]
   9 | 0.388328  [#######]
  10 | 0.365100  [#######]
  11 | 0.341049  [######]
  12 | 0.316178  [######]
  13 | 0.290498  [#####]
  14 | 0.264030  [#####]
  15 | 0.236807  [####]
  16 | 0.208877  [####]
  17 | 0.180299  [###]
  18 | 0.151151  [###]
  19 | 0.121527  [##]
  20 | 0.091539  [#]
  21 | 0.061317  [#]
  22 | 0.031010  []
  23 | 0.000784  []
  24 | -0.029175  []
  25 | -0.058666  []
  26 | -0.087473  []
  27 | -0.115367  []
  28 | -0.142112  []
  29 | -0.167464  []
  30 | -0.191181  []


In [14]:
%%writefile lattice_flow_final.c
#include <stdio.h>
#include <stdlib.h>
#include <complex.h>
#include <math.h>
#include <time.h>

#define N 4
#define STEPS 30
#define DT 0.05

typedef struct { double complex e[3][3]; } su3_matrix;

su3_matrix U[N][N][2];

// --- Math Helpers ---
su3_matrix multiply(su3_matrix A, su3_matrix B) {
    su3_matrix C = {{{0}}};
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) for(int k=0; k<3; k++)
        C.e[i][j] += A.e[i][k] * B.e[k][j];
    return C;
}

su3_matrix add(su3_matrix A, su3_matrix B) {
    su3_matrix C;
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) C.e[i][j] = A.e[i][j] + B.e[i][j];
    return C;
}

su3_matrix scale(double complex s, su3_matrix A) {
    su3_matrix C;
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) C.e[i][j] = s * A.e[i][j];
    return C;
}

su3_matrix dagger(su3_matrix A) {
    su3_matrix C;
    for(int i=0; i<3; i++) for(int j=0; j<3; j++) C.e[i][j] = conj(A.e[j][i]);
    return C;
}

su3_matrix project_to_su3_algebra(su3_matrix w) {
    su3_matrix z;
    for (int i = 0; i < 3; i++) for (int j = 0; j < 3; j++)
        z.e[i][j] = (w.e[i][j] - conj(w.e[j][i])) / 2.0;
    double complex tr = (z.e[0][0] + z.e[1][1] + z.e[2][2]) / 3.0;
    z.e[0][0] -= tr; z.e[1][1] -= tr; z.e[2][2] -= tr;
    return z;
}

su3_matrix exp_su3(su3_matrix Z) {
    su3_matrix Id = {{{1,0,0},{0,1,0},{0,0,1}}};
    su3_matrix term1 = add(Id, Z);
    su3_matrix term2 = scale(0.5, multiply(Z, Z));
    return add(term1, term2);
}

// --- Initialization ---
void init_lattice_safe() {
    su3_matrix Id = {{{1,0,0},{0,1,0},{0,0,1}}};
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            for(int dir=0; dir<2; dir++) {
                su3_matrix noise;
                for(int i=0;i<3;i++) for(int j=0;j<3;j++)
                    noise.e[i][j] = ((double)rand()/RAND_MAX - 0.5) + ((double)rand()/RAND_MAX - 0.5)*I;
                su3_matrix Z = project_to_su3_algebra(noise);
                su3_matrix Z_scaled = scale(1.0, Z);
                su3_matrix R = exp_su3(Z_scaled);
                U[x][y][dir] = multiply(R, Id);
            }
        }
    }
}

// --- Physics ---
double measure_plaquette() {
    double total_plaq = 0;
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            int x_next = (x+1)%N;
            int y_next = (y+1)%N;
            su3_matrix u1 = U[x][y][0];
            su3_matrix u2 = U[x_next][y][1];
            su3_matrix u3 = dagger(U[x][y_next][0]);
            su3_matrix u4 = dagger(U[x][y][1]);
            su3_matrix loop = multiply(multiply(multiply(u1, u2), u3), u4);
            total_plaq += creal(loop.e[0][0] + loop.e[1][1] + loop.e[2][2]);
        }
    }
    return total_plaq / (N*N * 3.0);
}

su3_matrix get_staple(int x, int y, int mu) {
    int nu = 1 - mu;
    int x_next_mu = (mu==0) ? (x+1)%N : x;
    int y_next_mu = (mu==1) ? (y+1)%N : y;
    int x_next_nu = (nu==0) ? (x+1)%N : x;
    int y_next_nu = (nu==1) ? (y+1)%N : y;
    su3_matrix A = U[x_next_mu][y_next_mu][nu];
    su3_matrix B = dagger(U[x_next_nu][y_next_nu][mu]);
    su3_matrix C = dagger(U[x][y][nu]);
    return multiply(multiply(A, B), C);
}

// --- FLOW STEP ---
void flow_step() {
    su3_matrix V[N][N][2];
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            for(int mu=0; mu<2; mu++) {
                su3_matrix staple = get_staple(x, y, mu);
                su3_matrix Omega = multiply(staple, dagger(U[x][y][mu]));
                su3_matrix Z = project_to_su3_algebra(Omega);

                // === THE FIX: FLIPPED SIGN ===
                // Changed scale(0.5 * DT, Z) to scale(-0.5 * DT, Z)
                V[x][y][mu] = exp_su3(scale(-0.5 * DT, Z));
            }
        }
    }
    for(int x=0; x<N; x++) {
        for(int y=0; y<N; y++) {
            for(int mu=0; mu<2; mu++) {
                U[x][y][mu] = multiply(V[x][y][mu], U[x][y][mu]);
            }
        }
    }
}

int main() {
    srand(time(NULL));
    init_lattice_safe();

    printf("=== FINAL 2D Lattice Gradient Flow ===\n");
    printf("Step | Avg Plaquette (Target: Rise to 1.0)\n");
    printf("------------------------------------------\n");

    for(int t=0; t<=STEPS; t++) {
        double p = measure_plaquette();

        printf("%4d | %.6f", t, p);
        int bar = (int)((p) * 20);
        if(bar<0) bar=0;
        printf("  [");
        for(int k=0; k<bar; k++) printf("#");
        printf("]\n");

        if(t < STEPS) flow_step();
    }
    return 0;
}

Overwriting lattice_flow_final.c


In [15]:
!gcc lattice_flow_final.c -o lattice_flow_final -lm && ./lattice_flow_final

=== FINAL 2D Lattice Gradient Flow ===
Step | Avg Plaquette (Target: Rise to 1.0)
------------------------------------------
   0 | 0.673078  [#############]
   1 | 0.682789  [#############]
   2 | 0.691920  [#############]
   3 | 0.700504  [##############]
   4 | 0.708571  [##############]
   5 | 0.716151  [##############]
   6 | 0.723270  [##############]
   7 | 0.729951  [##############]
   8 | 0.736215  [##############]
   9 | 0.742080  [##############]
  10 | 0.747563  [##############]
  11 | 0.752677  [###############]
  12 | 0.757432  [###############]
  13 | 0.761840  [###############]
  14 | 0.765908  [###############]
  15 | 0.769642  [###############]
  16 | 0.773050  [###############]
  17 | 0.776137  [###############]
  18 | 0.778908  [###############]
  19 | 0.781370  [###############]
  20 | 0.783531  [###############]
  21 | 0.785397  [###############]
  22 | 0.786978  [###############]
  23 | 0.788283  [###############]
  24 | 0.789323  [###############]
  25 | 0.79011

In [16]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
L = 32               # Lattice size (LxL)
STEPS = 50           # Number of Flow steps
DT = 0.05            # Step size (Keep small for stability)
SEED = 42            # Fixed seed for reproducibility

# === PHYSICS KERNEL ===

def get_plaquettes(links):
    """
    Calculates the field angle for every plaquette on the lattice.
    P_uv(x) = U_u(x) + U_v(x+u) - U_u(x+v) - U_v(x)
    """
    # link shape: (L, L, 2) -> 2 components are (mu=0 for x, mu=1 for y)

    # U_x(x,y)
    u0 = links[:, :, 0]
    # U_y(x+dx, y) -> Roll grid -1 in x-axis to look ahead
    u1_shift_x = np.roll(links[:, :, 1], -1, axis=0)
    # U_x(x, y+dy) -> Roll grid -1 in y-axis to look ahead
    u0_shift_y = np.roll(links[:, :, 0], -1, axis=1)
    # U_y(x, y)
    u1 = links[:, :, 1]

    # Plaquette angle: sum of links around the square
    # Note: angles are additive in U(1) (equivalent to multiplying matrices)
    return u0 + u1_shift_x - u0_shift_y - u1

def get_force(links):
    """
    Calculates the Gradient Flow force (derivative of Action).
    Action S = -Sum cos(P).
    Force = -dS/dLink.
    """
    # Get all plaquette angles P(x)
    P = get_plaquettes(links)

    # We need the sin(P) term for the derivative of -cos(P)
    sinP = np.sin(P)

    # Initialize force array matching links array structure
    force = np.zeros_like(links)

    # --- COMPONENT 0 (x-links) ---
    # An x-link at (x,y) contributes to:
    # 1. The plaquette at (x,y) [Positive contribution]
    # 2. The plaquette at (x, y-1) [Negative contribution]

    # Term 1: +sin(P) at (x,y)
    term1 = sinP
    # Term 2: -sin(P) at (x, y-1). Roll +1 on y-axis to look "behind"
    term2 = np.roll(sinP, 1, axis=1)

    # Flow Equation: dU/dt = -dS/dU
    # dS/dU ~ sin(P_up) - sin(P_down)
    # Therefore Force ~ - (sin(P_up) - sin(P_down)) = sin(P_down) - sin(P_up)
    force[:, :, 0] = term2 - term1

    # --- COMPONENT 1 (y-links) ---
    # A y-link at (x,y) contributes to:
    # 1. The plaquette at (x,y) [Negative contribution in standard orientation]
    # 2. The plaquette at (x-1, y) [Positive contribution]

    # Wait! In P = U_x(x) + U_y(x+x) - U_x(x+y) - U_y(x)
    # U_y(x) has a MINUS sign.
    # So dS/dU_y ~ -sin(P) * (-1) = sin(P)

    # Term 1 (Current plaquette): U_y(x) is the 4th term, has minus sign.
    # Derivative of -cos(P) wrt P is sin(P).
    # Derivative of P wrt U_y is -1.
    # Contribution: -sin(P)

    # Term 2 (Neighbor plaquette at x-1): U_y(x) is the 2nd term (U_y(x+dx)).
    # Contribution: +sin(P at x-1)

    term1_y = sinP # The plaquette where this link is the "left" side (minus)
    term2_y = np.roll(sinP, 1, axis=0) # The plaquette where this link is "right" side (plus)

    # Flow equation: -dS/dU
    # dS = sin(P_left)*(-1) + sin(P_right)*(+1)
    # Force = - ( sin(P_right) - sin(P_left) ) = sin(P_left) - sin(P_right)
    force[:, :, 1] = term1_y - term2_y

    return force

def calc_avg_plaquette(links):
    P = get_plaquettes(links)
    # Average of cosine of angles
    return np.mean(np.cos(P))

# === INTEGRATOR (Runge-Kutta 4) ===
# RK4 is much more stable than Euler. If this drops, the physics is stuck.
def rk4_step(links, dt):
    k1 = get_force(links)
    k2 = get_force(links + 0.5 * dt * k1)
    k3 = get_force(links + 0.5 * dt * k2)
    k4 = get_force(links + dt * k3)

    return links + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

# === MAIN SIMULATION ===

print(f"{'Step':<5} | {'Avg Plaquette (Target: 1.0)':<30}")
print("-" * 40)

# Initialize HOT (Random angles between -pi and pi)
np.random.seed(SEED)
links = np.random.uniform(-np.pi, np.pi, size=(L, L, 2))

history = []

for step in range(STEPS):
    # Measure
    avg_plaq = calc_avg_plaquette(links)
    history.append(avg_plaq)

    # Visual Progress Bar
    bar_len = int((avg_plaq - 0.5) * 50) # Scale for visual
    bar = "#" * max(0, bar_len)
    print(f"{step:<5} | {avg_plaq:.6f}  [{bar:<25}]")

    # Update
    links = rk4_step(links, DT)

    # Sanity Check for Divergence
    if np.isnan(avg_plaq):
        print("!!! SIMULATION EXPLODED (NaN detected) !!!")
        break

# Final Analysis
print("-" * 40)
if history[-1] > 0.99:
    print("SUCCESS: Reached Target (Vacuum State)")
elif history[-1] < history[-5]:
    print("FAILURE: Numerical instability detected (Action rising). Reduce DT.")
else:
    print(f"STALLED: Converged to {history[-1]:.4f}. Topological defects present.")

Step  | Avg Plaquette (Target: 1.0)   
----------------------------------------
0     | 0.009767  [                         ]
1     | 0.109518  [                         ]
2     | 0.206329  [                         ]
3     | 0.297496  [                         ]
4     | 0.381361  [                         ]
5     | 0.457171  [                         ]
6     | 0.524792  [#                        ]
7     | 0.584481  [####                     ]
8     | 0.636761  [######                   ]
9     | 0.682322  [#########                ]
10    | 0.721907  [###########              ]
11    | 0.756227  [############             ]
12    | 0.785928  [##############           ]
13    | 0.811601  [###############          ]
14    | 0.833793  [################         ]
15    | 0.853001  [#################        ]
16    | 0.869650  [##################       ]
17    | 0.884088  [###################      ]
18    | 0.896589  [###################      ]
19    | 0.907384  [####################     ]
